In [ ]:
### CRAG 의 데이터 중 web 결과인 search_result 를 잘 retrive 하는 retriver 만들기


In [ ]:
## 1. Base Retriver
    - parse_html()  :  beautifulSoup 으로 html -> text 로 변경
    - text-embedding-3-small 임베딩 모듈 사용
    - text_to_sentence_and_offset() : document 를 chunk 단위로 잘라줌
    - 전체 document 의 chunk embedding 과 query embedding 의 cosine 유사도 계산해서 top k chunk 리턴

class BaseRetriever:
    def __init__(self,):
        self.client = openai.OpenAI(api_key = os.environ["OPENAI_API_KEY"])

    def embed_text(self, texts):
        """Generate embeddings using OpenAI's embedding model."""
        if isinstance(texts, str):
            texts = [texts]

        response = self.client.embeddings.create(
            model="text-embedding-3-small",
            input=texts
        )

        # Extract embeddings correctly from the response object
        embeddings = [np.array(item.embedding) for item in response.data]  # Adjust based on actual attributes
        return np.array(embeddings)

    def retrieve(self, query, search_results, topk):
        # Get documents
        all_documents = parse_htmls(search_results)

        # Get chunks
        all_chunks = extract_chunks(all_documents)

        # Generate embeddings for all chunks and the query.
        all_embeddings = self.embed_text(all_chunks)
        query_embedding = self.embed_text(query)[0]  # Single query embedding

        # Calculate cosine similarity between query and sentence embeddings, and select the top sentences.
        cosine_scores = np.dot(all_embeddings, query_embedding) / (
            np.linalg.norm(all_embeddings, axis=1) * np.linalg.norm(query_embedding)
        )
        top_k_indices = (-cosine_scores).argsort()[:topk]
        top_k_chunks = np.array(all_chunks)[top_k_indices]

        return top_k_chunks

In [ ]:
## 2. Llama_index 를 이용한 Retriver class

Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

class LlamaIndexRetriever:
  def __init__(self):
      self.parser = SentenceSplitter( chunk_size=512, chunk_overlap=0 )

  def retrieve(self, query, search_results, topk):
      documents = []

      for document in parse_htmls(search_results):
        if not document:
            documents.append(Document(text=""))         # If no text is extracted, add an empty string as a placeholder.
        else:
            documents.append(Document(text=document))

      # Split documents into chunks & Create vector index
      base_index = VectorStoreIndex.from_documents( documents = documents, transformations=[self.parser] )

      # Execute query
      base_retriever = base_index.as_retriever( similarity_top_k=topk )
      retrieved_nodes = base_retriever.retrieve( query )
      retrieved_results = [ retrieved_node.node.get_content().strip() for retrieved_node in retrieved_nodes ]

      return retrieved_results

In [ ]:
## 3. 결과를 잘 말해줄 System Prompt 와 결과를 generation 할 Reader 클래스 정의

from openai import OpenAI
oai_client = OpenAI()

class Reader:
    def __init__(self):

        self.system_prompt = """
            You are provided with a question and various references.
            Your task is to answer the question succinctly, using the fewest words possible.
            If the references do not contain the necessary information to answer the question, respond with 'I don't know'.
            There is no need to explain the reasoning behind your answers.
        """

  def generate_response(self, query: str, top_k_chunks: list) -> str:
      """
      Generate answer from context.
      """
      llm_input = self.prompt_generator(query, top_k_chunks)
      completion = oai_client.chat.completions.create(
      model="gpt-3.5-turbo",
      temperature=0,
      messages=
      llm_input
      ).choices[0].message.content
      return completion

  def prompt_generator(self, query, top_k_chunks):
      user_message = ""
      references = ""

      if len(top_k_chunks) > 0:
          references += "# References \n"
          # Format the top sentences as references in the model's prompt template.
          for chunk_id, chunk in enumerate(top_k_chunks):
              references += f"- {chunk.strip()}\n"

      references = references[:MAX_CONTEXT_REFERENCES_LENGTH]
      # Limit the length of references to fit the model's input size.

      user_message += f"{references}\n------\n\n"
      user_message
      user_message += f"Using only the references listed above, answer the following question: \n"
      user_message += f"Question: {query}\n"

      llm_input = [
        {"role": "system", "content": self.system_prompt},
        {"role": "user", "content": user_message},
      ]

      return llm_input